ESERCIZIO

Sfida Bidirezionalità

Obbiettivo: implementare e confrontare una BiLSTM per il tagging semantico
Architettura: implementare un modello Keras con un layer Embedding e un layer Bidirectinal(LSTM(32))
Configurazione: impostareil merge_mode su 'sum' anzichè sul default 'concat'
Analisi: utilizza model.summary() per verificare se il numero di parametri è cambiato rispetto alla concatenazione
Verifica: addestra il modello su dataset fornito per 3 epoche e confronta l'accuratezza finale con un modello bidirezionale
Suggerimento: osserva attentamente se la somma degli stati (sum) parmette di mantenere lo stesso spazio di output del layer Dense finale.

In [1]:
import os

# =================================================================
# 1. SETUP DELL'AMBIENTE
# =================================================================
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers
import numpy as np

# --- CONFIGURAZIONI GLOBALI ---
VOCABOLARIO_SIZE = 10000
LUNGHEZZA_MAX = 150
CATEGORIE = 46
EPOCHS = 10
BATCH_SIZE = 128

# =================================================================
# 2. PREPARAZIONE DATI
# =================================================================
def prepara_dataset():
    print("[1] Caricamento Dataset Reuters...")
    (x_train, y_train), (x_test, y_test) = keras.datasets.reuters.load_data(num_words=VOCABOLARIO_SIZE)
    
    x_train = keras.utils.pad_sequences(x_train, maxlen=LUNGHEZZA_MAX)
    x_test = keras.utils.pad_sequences(x_test, maxlen=LUNGHEZZA_MAX)
    
    y_train = keras.utils.to_categorical(y_train, CATEGORIE)
    y_test = keras.utils.to_categorical(y_test, CATEGORIE)
    
    return (x_train, y_train), (x_test, y_test)

# =================================================================
# 3. COSTRUZIONE MODELLI
# =================================================================

def build_model(mode="uni", merge_mode=None):
    """
    Crea un modello basato sulla configurazione richiesta.
    mode: 'uni' per Unidirezionale, 'bi' per Bidirezionale.
    merge_mode: 'sum', 'concat', etc. (solo per 'bi').
    """
    inputs = keras.Input(shape=(LUNGHEZZA_MAX,))
    x = layers.Embedding(VOCABOLARIO_SIZE, 128)(inputs)
    
    if mode == "bi":
        # LSTM a 32 unità bidirezionale
        x = layers.Bidirectional(layers.LSTM(32), merge_mode=merge_mode)(x)
    else:
        # LSTM a 32 unità unidirezionale
        x = layers.LSTM(32)(x)
    
    outputs = layers.Dense(CATEGORIE, activation="softmax")(x)
    
    name = f"Modello_{mode}"
    if merge_mode: name += f"_{merge_mode}"
    
    model = keras.Model(inputs, outputs, name=name)
    model.compile(optimizer="adamw", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# =================================================================
# 4. ESECUZIONE SFIDA
# =================================================================
def main():
    (x_train, y_train), (x_test, y_test) = prepara_dataset()
    
    # --- ANALISI PARAMETRICI ---
    print("\n" + "="*50)
    print("ANALISI ARCHITETTURALE")
    print("="*50)
    
    model_sum = build_model(mode="bi", merge_mode="sum")
    model_concat = build_model(mode="bi", merge_mode="concat") # Default
    
    print("\n--- MODELLO BI-LSTM (SUM) ---")
    model_sum.summary()
    
    print("\n--- MODELLO BI-LSTM (CONCAT) ---")
    model_concat.summary()
    
    # NOTA: Il numero di parametri dello strato Bidirectional è IDENTICO.
    # Tuttavia, lo strato Dense successivo ha meno parametri in 'sum' perché 
    # riceve 32 input invece di 64 (32+32).
    
    # --- ADDESTRAMENTO E CONFRONTO ---
    print(f"ADDESTRAMENTO ({EPOCHS} EPOCHE)")
    print("="*50)
    
    # Modello Bidirezionale (SUM)
    print("\nTraining Bi-LSTM (merge_mode='sum')...")
    history_bi = model_sum.fit(
        x_train, y_train, 
        epochs=EPOCHS, 
        batch_size=BATCH_SIZE, 
        validation_split=0.1, 
        verbose=1
    )
    acc_bi = model_sum.evaluate(x_test, y_test, verbose=0)[1]
    
    # Modello Unidirezionale
    model_uni = build_model(mode="uni")
    print("\nTraining Unidirectional LSTM...")
    history_uni = model_uni.fit(
        x_train, y_train, 
        epochs=EPOCHS, 
        batch_size=BATCH_SIZE, 
        validation_split=0.1, 
        verbose=1
    )
    acc_uni = model_uni.evaluate(x_test, y_test, verbose=0)[1]
    
    # --- RISULTATI FINALI ---
    print("\n" + "="*50)
    print("CONFRONTO FINALE")
    print("="*50)
    print(f"Accuratezza Bi-LSTM (Sum): {acc_bi:.2%}")
    print(f"Accuratezza Uni-LSTM:      {acc_uni:.2%}")
    
    improvement = ((acc_bi - acc_uni) / acc_uni) * 100
    print(f"\nIl modello Bidirezionale ha un incremento di performance del {improvement:+.2f}% rispetto all'unidirezionale.")
    
    # Verifica dello spazio di output (usando la proprietà output del layer che è corretta in Keras 3)
    try:
        output_dim = model_sum.layers[2].output.shape[-1]
        if output_dim == 32:
            print("\nVerifica Output Space: Corretto. 'sum' mantiene 32 unità in uscita,")
            print("consentendo di usare la stessa dimensione dello strato LSTM originale.")
    except Exception:
        print("\nNota: Impossibile verificare dinamicamente lo shape, controllare model.summary().")

if __name__ == "__main__":
    main()

[1] Caricamento Dataset Reuters...

ANALISI ARCHITETTURALE

--- MODELLO BI-LSTM (SUM) ---


Model: "Modello_bi_sum"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 150, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 32)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 46)             │         1,518 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,322,734 (5.05 MB)

 Trainable params: 1,322,734 (5.05 MB)

...

ADDESTRAMENTO (10 EPOCHE)

Training Bi-LSTM (merge_mode='sum')...
Epoch 1/10


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:1076: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n, c_n = torch._VF.lstm(


64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.3506 - loss: 2.7808 - val_accuracy: 0.4116 - val_loss: 2.1773
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5041 - loss: 1.9335 - val_accuracy: 0.5328 - val_loss: 1.8644
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5813 - loss: 1.6896 - val_accuracy: 0.5662 - val_loss: 1.7736
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6159 - loss: 1.5916 - val_accuracy: 0.5473 - val_loss: 1.9465
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6364 - loss: 1.5244 - val_accuracy: 0.5951 - val_loss: 1.6882
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6708 - loss: 1.3810 - val_accuracy: 0.6207 - val_loss: 1.6355
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6759 - loss: 1.2944 - val_accuracy: 0.6274 - val_loss: 1.6021
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7051 - loss: 1.1495 - val_accuracy: 0.6352 - val_loss: 1.

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:656: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n, c_n = torch._VF.lstm(


64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3407 - loss: 2.9768 - val_accuracy: 0.3315 - val_loss: 2.4826
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4185 - loss: 2.3184 - val_accuracy: 0.4894 - val_loss: 2.2852
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5255 - loss: 2.1041 - val_accuracy: 0.4994 - val_loss: 2.0804
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5378 - loss: 1.8962 - val_accuracy: 0.5217 - val_loss: 1.9101
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5807 - loss: 1.7111 - val_accuracy: 0.5595 - val_loss: 1.7849
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6172 - loss: 1.5617 - val_accuracy: 0.5895 - val_loss: 1.7357
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6446 - loss: 1.4365 - val_accuracy: 0.6096 - val_loss: 1.6499
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6816 - loss: 1.2964 - val_accuracy: 0.6085 - val_loss: 1.